# Modelo XGBoost
---

### Configuración de ambiente

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

# Carga de modulos_apex
from modulos_apex_dev import BTC_DataExtractor as btc_etl
from modulos_apex_dev import trading_backtester as tbt

import os
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

CURRENT_DIR = os.getcwd()

## Carga de datos
---


In [2]:
CURRENT_DIR

'/Users/julesaccm/Documents/Repos/Apex-project/notebooks'

In [11]:
# Cargamos los datos desde la carpeta data y si no existen, ejecutamos el pipeline
try: 
    os.chdir('..')
    os.chdir('data/processed')
    df_final = pd.read_csv('btc_data.csv', index_col=0)
    print("Datos cargados desde 'data/processed/btc_data.csv'")

except FileNotFoundError:
    print("Archivo no encontrado. Ejecutando el pipeline para generar los datos...")

    # 1. Instanciar la clase (Configuramos las variables globales del proceso)
    extractor = btc_etl.BTC_DataExtractor(
        fecha_inicio="2024-01-01", 
        fecha_fin="2026-01-01", 
        ventana_critica=5
    )

    # 2. Ejecutar las transformaciones en cadena (Pipeline)
    # Cada paso toma el DataFrame del paso anterior, lo transforma y lo devuelve

    # Paso A: Obtener el precio histórico y volumen
    df_base = extractor.obtener_datos_btc()

    # Paso B: Etiquetar máximos y mínimos (Nuestro Target)
    df_etiquetado = extractor.etiquetar_puntos_criticos(df_base)

    # Paso C: Calcular todo el análisis técnico
    df_con_indicadores = extractor.agregar_indicadores_avanzados(df_etiquetado)

    # Paso D: Enriquecer con datos macroeconómicos
    df_final = extractor.agregar_contexto_macro(df_con_indicadores)

    # 3. Revisar el resultado final
    print("\n--- Vista previa de las primeras 5 filas ---")
    print(df_final.head())

    # Creamos el archivo en la carpeta data/processed
    os.chdir(CURRENT_DIR)
    os.chdir('..')
    os.makedirs('data/processed', exist_ok=True)
    df_final.to_csv('data/processed/btc_data.csv', index=True)
    
    print("Datos generados y guardados en 'data/processed/btc_data.csv'")

finally:
    os.chdir(CURRENT_DIR) 


Archivo no encontrado. Ejecutando el pipeline para generar los datos...
Descargando datos de BTC-USD desde 2024-01-01...


[*********************100%***********************]  1 of 1 completed


Iniciando descarga de datos macroeconómicos...
-> Descargando SP500 (^GSPC)...
-> Descargando DXY (DX-Y.NYB)...
-> Descargando Oro (GC=F)...

¡Contexto macro agregado exitosamente!

--- Vista previa de las primeras 5 filas ---
                    Open          High           Low         Close  \
Date                                                                 
2024-02-04  42994.941406  43097.644531  42374.832031  42583.582031   
2024-02-05  42577.621094  43494.250000  42264.816406  42658.667969   
2024-02-06  42657.390625  43344.148438  42529.019531  43084.671875   
2024-02-07  43090.019531  44341.949219  42775.957031  44318.222656   
2024-02-08  44332.125000  45575.839844  44332.125000  45301.566406   

                 Volume  Retorno_Log  Target     RSI_14  BBL_20_2.0_2.0  \
Date                                                                      
2024-02-04  14802225490    -0.009551       0  55.156994    39455.992843   
2024-02-05  18715487317     0.001762       0  55.579270  

In [12]:
df_final.head()

,Open,High,Low,Close,Volume,Retorno_Log,Target,RSI_14,BBL_20_2.0_2.0,BBM_20_2.0_2.0,...,ROC_10,BULLP_13,BEARP_13,UO_7_14_28,SP500_Close,DXY_Close,Oro_Close,SP500_Retorno,DXY_Retorno,Oro_Retorno
Date,,,,,,,,,,,,,,,,,,,,,
2024-02-04,42994.941406,43097.644531,42374.832031,42583.582031,14802225490,-0.009551,0,55.156994,39455.992843,41899.333984,...,6.635414,619.166722,-103.645778,54.454136,4942.810059,104.449997,2025.699951,0.000000,0.000000,0.000000
2024-02-05,42577.621094,43494.250000,42264.816406,42658.667969,18715487317,0.001762,0,55.579270,39475.186244,41874.520117,...,2.013056,990.030739,-239.402854,47.903907,4942.810059,104.449997,2025.699951,0.000000,0.000000,0.000000
2024-02-06,42657.390625,43344.148438,42529.019531,43084.671875,16798476726,0.009937,0,57.996020,39461.555932,41891.621094,...,2.290161,757.007375,-58.121531,51.256441,4954.229980,104.180000,2034.500000,0.002310,-0.002585,0.004344
2024-02-07,43090.019531,44341.949219,42775.957031,44318.222656,21126587775,0.028229,0,64.088663,39405.656914,42044.429297,...,5.430229,1507.510786,-58.481402,62.110142,4995.060059,104.040001,2035.199951,0.008241,-0.001344,0.000344
2024-02-08,44332.125000,45575.839844,44332.125000,45301.566406,26154524080,0.021946,0,68.065262,39225.992581,42228.587305,...,4.650963,2388.954557,1145.239714,61.831010,4997.910156,104.150002,2032.199951,0.000571,0.001057,-0.001474


## Entrenamiento del modelo XGBoost
---

In [13]:
# --- 1. CONFIGURACIÓN DE FEATURES ---

df_apex = df_final.copy()

# Excluimos los precios crudos de BTC, el Target y los precios crudos macroeconómicos
columnas_excluir = [
    'Open', 'High', 'Low', 'Close', 'Volume', 'Target',
    'SP500_Close', 'DXY_Close', 'Oro_Close' # ¡Importante! Solo queremos sus retornos
    , 'Retorno_Log'
]

# Creamos la lista final de features (X)
features = [col for col in df_apex.columns if col not in columnas_excluir]

# --- 2. PREPARACIÓN DE DATOS (Train / Test) ---
X = df_apex[features]
y = df_apex['Target']

# División cronológica (80% Train, 20% Test)
split_idx = int(len(df_apex) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Entrenando con {len(features)} variables (Técnicas + Macro)...")
print(f"Días de entrenamiento: {len(X_train)} | Días de prueba: {len(X_test)}")

Entrenando con 30 variables (Técnicas + Macro)...
Días de entrenamiento: 557 | Días de prueba: 140


In [14]:
print("--- INICIANDO ENTRENAMIENTO XGBOOST ---")

# 1. Traducir las etiquetas (-1, 0, 1) a formato XGBoost (0, 1, 2)
le = LabelEncoder()
y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.transform(y_test)

# 2. Configurar el modelo XGBoost
# Nota: XGBoost es muy potente, por lo que usamos max_depth bajo para que no memorice el pasado
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,         # Árboles menos profundos para generalizar mejor
    learning_rate=0.05,  # Aprende despacio para no sobreajustarse
    random_state=42,
    n_jobs=-1
)

# 3. Entrenar
print("Entrenando modelo XGBoost...")
modelo_xgb.fit(X_train, y_train_xgb)

# 4. Obtener las probabilidades en los datos de prueba
probabilidades_xgb = modelo_xgb.predict_proba(X_test)

# 5. Aplicar nuestro "Umbral de Valentía" (25%)
UMBRAL_EXTREMO = 0.25
y_pred_xgb_ajustado = np.zeros(len(y_test))

# Encontramos qué columna de probabilidad corresponde al Mínimo y al Máximo
idx_minimo = np.where(le.classes_ == -1)[0][0]
idx_maximo = np.where(le.classes_ == 1)[0][0]

for i in range(len(probabilidades_xgb)):
    prob_min = probabilidades_xgb[i][idx_minimo]
    prob_max = probabilidades_xgb[i][idx_maximo]
    
    if prob_min >= UMBRAL_EXTREMO:
        y_pred_xgb_ajustado[i] = -1
    elif prob_max >= UMBRAL_EXTREMO:
        y_pred_xgb_ajustado[i] = 1


--- INICIANDO ENTRENAMIENTO XGBOOST ---
Entrenando modelo XGBoost...


In [15]:
# 5. Recuperamos la porción de "Test" pero del DataFrame original (que SÍ tiene 'Close')
split_idx = int(len(df_apex) * 0.8)
df_test_completo = df_apex.iloc[split_idx:].copy()

assert len(df_test_completo) == len(y_pred_xgb_ajustado), "Error: Las longitudes no coinciden"

## Tunning
---

In [16]:
# Inicializo el backtester
simulador = tbt.Backtester(df_test_completo, capital_inicial=1000.0)

In [17]:
from sklearn.model_selection import ParameterGrid
import xgboost as xgb
import numpy as np

print("Iniciando Hyperparameter Tuning Financiero para XGBoost...")

# 1. Definimos la malla de hiperparámetros a explorar
param_grid = {
    'max_depth': [3, 4, 6],               # Qué tan complejos son los árboles
    'learning_rate': [0.01, 0.05, 0.1],   # Qué tan rápido aprende
    'n_estimators': [50, 100],            # Cantidad de árboles
    'umbral': [0.20, 0.25, 0.30]          # El umbral de probabilidad para decidir comprar/vender
}

grid = ParameterGrid(param_grid)
mejor_retorno = -np.inf
mejores_parametros = None
mejor_historial = None

total_combinaciones = len(grid)
print(f"Total de combinaciones a evaluar: {total_combinaciones}")

for i, params in enumerate(grid):
    # A. Entrenamos XGBoost con los parámetros actuales
    modelo_tuning = xgb.XGBClassifier(
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        n_estimators=params['n_estimators'],
        random_state=42,
        n_jobs=-1
    )
    
    # Asumimos que y_train_xgb y X_train ya están listos del paso anterior
    modelo_tuning.fit(X_train, y_train_xgb)
    
    # B. Predecimos probabilidades
    probabilidades = modelo_tuning.predict_proba(X_test)
    
    # C. Aplicamos el Umbral dinámico de la malla
    y_pred_tuning = np.zeros(len(y_test))
    for j in range(len(probabilidades)):
        if probabilidades[j][idx_minimo] >= params['umbral']:
            y_pred_tuning[j] = -1
        elif probabilidades[j][idx_maximo] >= params['umbral']:
            y_pred_tuning[j] = 1
            
    # D. Evaluamos en el simulador financiero (Usamos el Trailing Stop)
    # Por ahora dejamos los parámetros del trailing fijos, para solo medir al modelo
    retorno, trades = simulador.backtest_con_trailing_stop(
        y_pred_tuning, 
        stop_loss_inicial=0.05, 
        trailing_activation=0.04, 
        trailing_distancia=0.02,
        graficar=False # Apagado para que no imprima 54 gráficas
    )
    
    # E. Guardamos el campeón
    if retorno > mejor_retorno:
        mejor_retorno = retorno
        mejores_parametros = params
        mejor_historial = trades

print("\n" + "="*50)
print(f"🏆 MEJOR COMBINACIÓN ENCONTRADA")
print("="*50)
print(f"Retorno Total: {mejor_retorno:.2f}%")
print("Parámetros del Modelo y Umbral:")
for k, v in mejores_parametros.items():
    print(f" - {k}: {v}")
print(f"Operaciones realizadas: {len(mejor_historial[mejor_historial['Tipo'].str.contains('Venta')]) if not mejor_historial.empty else 0}")

Iniciando Hyperparameter Tuning Financiero para XGBoost...
Total de combinaciones a evaluar: 54
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $989.30 USD
Retorno Total: -1.07%
Total de operaciones cerradas: 13
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1003.57 USD
Retorno Total: 0.36%
Total de operaciones cerradas: 11
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $998.24 USD
Retorno Total: -0.18%
Total de operaciones cerradas: 9
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1003.57 USD
Retorno Total: 0.36%
Total de operaciones cerradas: 11
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $997.10 USD
Retorno Total: -0.29%
Total de operaciones cerradas: 11
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1000.29 USD
Retorno Total: 0.03%
Total de operaciones cerradas: 7
--- RESULTADOS DEL BACKTEST ---
Capi

### Optimización de la gestión del riesgo

In [20]:
print("--- INICIANDO OPTIMIZACIÓN DEL ESCUDO (TRAILING STOP) ---")

# 1. ENTRENAMOS EL MODELO CAMPEÓN (Solo 1 vez)
modelo_campeon = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.05,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Entrenando el XGBoost Campeón...")
modelo_campeon.fit(X_train, y_train_xgb)
probabilidades = modelo_campeon.predict_proba(X_test)

# Aplicamos el umbral fijo de 0.3
y_pred_campeon = np.zeros(len(y_test))
for j in range(len(probabilidades)):
    if probabilidades[j][idx_minimo] >= 0.3:
        y_pred_campeon[j] = -1
    elif probabilidades[j][idx_maximo] >= 0.3:
        y_pred_campeon[j] = 1

# 2. DEFINIMOS LA MALLA DE GESTIÓN DE RIESGO
param_grid_riesgo = {
    'tamaño_posicion': [0.10, 0.20, 0.30],        # Invertir el 10%, 20% o 30% del capital
    'stop_loss_inicial': [0.03, 0.05, 0.08],      # Límite de pérdida inicial
    'trailing_activation': [0.03, 0.05, 0.10],    # % de subida para "despertar" el trailing
    'trailing_distancia': [0.015, 0.03, 0.05]     # Distancia a la que persigue al precio
}

grid_riesgo = ParameterGrid(param_grid_riesgo)
mejor_retorno_riesgo = -np.inf
mejores_params_riesgo = None
mejor_historial_riesgo = None

print(f"Total de combinaciones de riesgo a evaluar: {len(grid_riesgo)}")

# Instanciamos tu clase Backtester

# 3. BUCLE DE OPTIMIZACIÓN DEL BACKTESTER
for params in grid_riesgo:
    # FILTRO LÓGICO: La distancia del trailing debe ser menor a la activación.
    # Si pedimos que se active al 3% y que la distancia sea del 5%, nos sacaría inmediatamente.
    if params['trailing_distancia'] >= params['trailing_activation']:
        continue 
        
    retorno, trades_df = simulador.backtest_con_trailing_stop(
        predicciones=y_pred_campeon,
        tamaño_posicion=params['tamaño_posicion'],
        stop_loss_inicial=params['stop_loss_inicial'],
        trailing_activation=params['trailing_activation'],
        trailing_distancia=params['trailing_distancia'],
        graficar=False
    )
    
    if retorno > mejor_retorno_riesgo:
        mejor_retorno_riesgo = retorno
        mejores_params_riesgo = params
        mejor_historial_riesgo = trades_df

print("\n" + "="*50)
print(f"🛡️ MEJOR CONFIGURACIÓN DE GESTIÓN DE RIESGO")
print("="*50)
print(f"Retorno Total: {mejor_retorno_riesgo:.2f}%")
print("Parámetros del Trailing Stop:")
for k, v in mejores_params_riesgo.items():
    print(f" - {k}: {v}")
print(f"Operaciones cerradas: {len(mejor_historial_riesgo[mejor_historial_riesgo['Tipo'].str.contains('Venta')]) if not mejor_historial_riesgo.empty else 0}")

--- INICIANDO OPTIMIZACIÓN DEL ESCUDO (TRAILING STOP) ---
Entrenando el XGBoost Campeón...
Total de combinaciones de riesgo a evaluar: 81
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1013.53 USD
Retorno Total: 1.35%
Total de operaciones cerradas: 5
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1009.29 USD
Retorno Total: 0.93%
Total de operaciones cerradas: 5
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1006.24 USD
Retorno Total: 0.62%
Total de operaciones cerradas: 6
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1012.56 USD
Retorno Total: 1.26%
Total de operaciones cerradas: 5
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1011.12 USD
Retorno Total: 1.11%
Total de operaciones cerradas: 6
--- RESULTADOS DEL BACKTEST ---
Capital Inicial: $1000.00 USD
Capital Final: $1008.83 USD
Retorno Total: 0.88%
Total de operaciones cerradas: 